# PII 마스킹

명시적으로 만든 합성 개인정보를 모델 호출과 감사 로그 전에 가립니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 탐지 함수는 합성 이메일과 전화번호 패턴의 시작·끝 위치를 찾고, 마스킹 함수는 실제 문자열을 [EMAIL]·[PHONE] 같은 종류 표지로 바꿉니다. 모델 호출과 로그 저장에는 마스킹된 문자열만 전달하고 원문은 출력하지 않습니다.

**다른 상황에 적용.** 주민등록번호가 필요 없는 문의 분류에서는 [ID_NUMBER] 표지만 남겨도 업무 유형을 판단할 수 있습니다. 반대로 배송 연락처럼 실제 번호가 필요한 후속 시스템에는 모델을 거치지 않는 별도 권한 경로를 설계해야 합니다.

**비교와 경계.** 정규식은 알려진 모양을 찾을 뿐 사람 이름·주소·우회 표기를 모두 탐지하지 못합니다. 마스킹은 암호화나 접근 통제를 대신하지 않으며, 이미 외부로 전송한 뒤 적용하면 보호 효과가 없습니다.

**이 셀의 코드 읽기.** 이 셀은 앞 셀에서 만든 값을 입력으로 읽어 위에서 아래 순서로 처리하고, 왼쪽 변수에 다음 셀이 사용할 결과를 저장합니다.


In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

실제 사람의 정보는 사용하지 않습니다. 예상 API 비용은 0.01달러 미만입니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [2]:
from pathlib import Path
import os, sys
from dotenv import find_dotenv, load_dotenv

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part4/ch16-guardrails").is_dir())
LAB = ROOT / "labs/part4/ch16-guardrails"
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
print("Python:", sys.version.split()[0], "| API 키 준비: True")

Python: 3.12.13 | API 키 준비: True


### 다음 코드 셀 읽기

**이 셀의 코드 읽기.** `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5.4-mini", temperature=0, max_tokens=140,
    timeout=30, max_retries=0,
)
print("모델 준비:", type(model).__name__)

모델 준비: ChatOpenAI


## 기본 실행

PII(personally identifiable information)는 개인을 알아볼 수 있게 하는 정보입니다.

**이 셀의 코드 읽기.** `mask_pii(text: str)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
import re

def mask_pii(text: str) -> tuple[str, dict[str, int]]:
    patterns = {
        "email": r"(?<![\w.])[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}(?![\w.])",
        "phone": r"(?<!\d)01[016789][ -]?\d{3,4}[ -]?\d{4}(?!\d)",
    }
    masked, counts = text, {}
    for name, pattern in patterns.items():
        masked, counts[name] = re.subn(pattern, f"<{name.upper()}_MASKED>", masked)
    return masked, counts

synthetic = "문의자는 test.user@example.com, 010-1234-5678로 답변을 요청했습니다."
masked, counts = mask_pii(synthetic)
print("마스킹:", masked)
print("종류별 개수:", counts)

마스킹: 문의자는 <EMAIL_MASKED>, <PHONE_MASKED>로 답변을 요청했습니다.
종류별 개수: {'email': 1, 'phone': 1}


## 모델 입력 확인

모델에는 마스킹된 문장만 전달합니다.

**이 셀의 코드 읽기.** `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [5]:
response = model.invoke([
    {"role": "system", "content": "입력에 남은 연락 수단의 종류만 말하세요."},
    {"role": "user", "content": masked},
])
print("모델 답:", response.text)
print("원문 이메일 재노출:", "test.user@example.com" in response.text)

모델 답: 이메일, 전화번호
원문 이메일 재노출: False


## 실패 재현

원문을 로그에 쓰면 개인 식별 문자열이 그대로 남습니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [6]:
unsafe_log = {"query": synthetic}
print("원문 이메일 포함:", "test.user@example.com" in unsafe_log["query"])
assert "test.user@example.com" in unsafe_log["query"]

원문 이메일 포함: True


## 수정

마스킹 결과와 탐지 개수만 기록합니다.

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [7]:
safe_log = {"query_masked": masked, "pii_counts": counts}
print(safe_log)
assert "test.user@example.com" not in __import__('json').dumps(safe_log)

{'query_masked': '문의자는 <EMAIL_MASKED>, <PHONE_MASKED>로 답변을 요청했습니다.', 'pii_counts': {'email': 1, 'phone': 1}}
